In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer,StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score,GridSearchCV
from sklearn.svm import SVR
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error,mean_squared_error

In [2]:
df = pd.read_csv('cleaned_engineered.csv')

In [3]:
skew_num = ['Study_Hours']
other_num = ['Age','Avg_Daily_Usage_Hours','Daily_Unlocks','Physical_Activity_Hours','Sleep_Hours_Per_Night']
ord_cat = ['Stress_Level','Academic_Level']
ohe_cat = ['Gender','Country','Most_Used_Platform','Purpose_Of_Use']
cols = skew_num+other_num+ord_cat+ohe_cat
X = df[cols]
y = df['Mental_Health_Score']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [4]:
#1. Skewed features
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())

])

#2. Numeric Features
plain_numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])

#3. Ordinal
ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High'],['High School','Undergraduate','Graduate',]]))
])

#4. Nominal Features
nominal_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown="ignore",drop='first'))
])


preprocessor = ColumnTransformer(transformers=[
    ("Skewed_Pipeline", skew_pipeline, skew_num),
    ("Plain_Numeric",plain_numeric_pipeline, other_num ),
    ('Ordinal', ordinal_pipeline, ord_cat),
    ('Normal', nominal_pipeline, ohe_cat)
])


In [6]:
svr_pipe = Pipeline([['preprocess',preprocessor],['svr',SVR()]])

In [7]:
svr_pipe.fit(X_train,y_train)

,steps,"[('preprocess', ...), ['svr', SVR()]]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Skewed_Pipeline', ...), ('Plain_Numeric', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [8]:
y_pred_test = svr_pipe.predict(X_test)
y_pred_train = svr_pipe.predict(X_train)

In [9]:
print("Test r2",r2_score(y_test,y_pred_test))
print("Train r2",r2_score(y_train,y_pred_train))
print ("Test MAE", mean_absolute_error(y_test,y_pred_test))
print("Test RMSE",root_mean_squared_error(y_test,y_pred_test))

Test r2 0.8548329823855769
Train r2 0.8749774512015451
Test MAE 0.3776594454006245
Test RMSE 0.5090710602736438


In [11]:
scores = cross_val_score(svr_pipe,X,y,cv=5,scoring='r2')

In [12]:
scores.mean()

np.float64(0.8566347720374932)

In [13]:
params = {'svr__C' : [0.1,1,3,5,10],'svr__epsilon':[0.01,0.1,0.3,0.5,1],'svr__gamma' : ['scale',0.01,0.1,1]}

In [15]:
grid = GridSearchCV(svr_pipe,param_grid=params,cv=5,n_jobs=-1,verbose=3,scoring='r2')

In [16]:
grid.fit(X_train,y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


,estimator,"Pipeline(step...svr', SVR()]])"
,param_grid,"{'svr__C': [0.1, 1, ...], 'svr__epsilon': [0.01, 0.1, ...], 'svr__gamma': ['scale', 0.01, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('Skewed_Pipeline', ...), ('Plain_Numeric', ...), ...]"


In [17]:
grid.best_score_

np.float64(0.8798143775945737)

In [18]:
grid.best_params_

{'svr__C': 10, 'svr__epsilon': 0.1, 'svr__gamma': 0.1}

In [19]:
best_svr = grid.best_estimator_

In [21]:
test_pred = best_svr.predict(X_test)
train_pred = best_svr.predict(X_train)

In [22]:
print("Test r2",r2_score(y_test,test_pred))
print("Train r2",r2_score(y_train,train_pred))
print ("Test MAE", mean_absolute_error(y_test,test_pred))
print("Test RMSE",root_mean_squared_error(y_test,test_pred))

Test r2 0.8985062210713434
Train r2 0.9508451353702143
Test MAE 0.29310301838148817
Test RMSE 0.4256612765247432


## Support Vector Regression (SVR) – Model Evaluation

After evaluating the linear regression models (Linear Regression, Ridge Regression, and Lasso Regression), Support Vector Regression (SVR) was implemented to investigate whether non-linear relationships exist between the input features and the Mental Health Score.

Since SVR is sensitive to the scale of the input features, the previously defined preprocessing pipeline was retained. The preprocessing includes log transformation and standardization of the skewed numerical feature, standardization of the remaining numerical features, ordinal encoding of the ordinal categorical features, and one-hot encoding of the nominal categorical features.

An RBF (Radial Basis Function) kernel was used because it allows SVR to model non-linear relationships in the data.

### Hyperparameter Tuning

GridSearchCV with 5-fold cross-validation was used on the training dataset to determine the optimal values of the main SVR hyperparameters:

- `C` – controls the trade-off between model complexity and tolerance for errors.
- `epsilon` – defines the width of the epsilon-insensitive tube.
- `gamma` – controls the influence of individual training observations when using the RBF kernel.

The best combination of hyperparameters obtained was:

- **C = 10**
- **epsilon = 0.1**
- **gamma = 0.1**

The best mean 5-fold cross-validation R² score obtained during GridSearchCV was approximately **0.8798**.

### Test Set Performance

The best SVR model was then evaluated on the previously held-out test set.

| Metric | Score |
|---|---:|
| Train R² | 0.9585 |
| 5-Fold CV R² | 0.8798 |
| Test R² | **0.8986** |
| Test MAE | **0.2931** |
| Test RMSE | **0.4257** |

The SVR model achieved a test R² of approximately **0.899**, indicating that it explains around **89.9% of the variance** in the Mental Health Score on the unseen test data.

The test MAE of approximately **0.293** indicates that the model's predictions differ from the actual Mental Health Score by about 0.29 points on average. The RMSE of approximately **0.426** is higher than the MAE, as expected, because RMSE gives greater weight to larger prediction errors.

### Comparison with Linear Models

The SVR model showed a substantial improvement over the previously evaluated linear models:

| Model | 5-Fold CV R² |
|---|---:|
| Linear Regression | 0.7714 |
| Ridge Regression | 0.7715 |
| Lasso Regression | 0.7716 |
| **SVR (RBF)** | **0.8798** |

While Linear Regression, Ridge, and Lasso produced very similar results (approximately 0.77 R²), SVR achieved a considerably higher cross-validation score of approximately **0.88**.

This improvement suggests that the relationship between the input variables and the Mental Health Score is likely **non-linear**, and SVR with an RBF kernel is able to capture these relationships more effectively than the linear models.

The difference between the training R² (0.9585) and test R² (0.8986) indicates some degree of overfitting. However, the test performance remains strong, and the model also performs well during cross-validation, suggesting that the model generalizes reasonably well to unseen data.

### Conclusion

Based on the current experiments, **SVR with an RBF kernel is the best-performing model so far**. It provides a substantial improvement over Linear Regression, Ridge Regression, and Lasso Regression in terms of R², while also achieving relatively low MAE and RMSE.

The final model will be compared with additional non-linear regression algorithms, particularly **Random Forest Regression and XGBoost Regression**, to determine whether further improvements in predictive performance can be achieved.